# 12 — Mechanism: which regulators and pathways sit behind the aging genes?

09 gave, per tissue × cell type, a DESeq2 statistic for every gene with mice as samples. Here those statistics are read through prior knowledge:

| Question | Resource | Method |
| --- | --- | --- |
| Which **transcription factors** change activity? | CollecTRI: 1,165 mouse TFs and their signed targets | univariate linear model (`decoupler.mt.ulm`): does a TF's target set move together? |
| Which **signaling pathways**? | PROGENy: 14 pathways, footprint genes | multivariate linear model (`mlm`) |
| Which **hallmark processes**? | MSigDB Hallmark, 50 sets | `ulm` |
| Which **cell–cell signals** change? | liana mouse consensus ligand–receptor list | ligand change in the sender × receptor change in the receiver, both from 09 (mice as samples) |

Rows: the 22 robust rows (age signal, not technically confounded) tested in 09, with the rest for contrast. Activities are t-values: > 2 or < −2 is a clear change. A TF or pathway is called **shared** when it moves the same way in rows from at least 4 tissues. The planned human check (GTEx) needs a large separate download and is left out of this phase.

### 1. Setup

In [ ]:
import sys
sys.path.append("..")          # so "src" can be imported from the notebook folder

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import decoupler as dc
import liana as li
from src import aging as ag

A = ag.paths("atlas")
atlas = pd.read_csv(A.figure("atlas.csv"))
atlas["robust"] = atlas["verdict"].eq("age signal") & ~atlas["tech_confounded"].astype(bool)
atlas["row"] = atlas["tissue"] + " | " + atlas["cell_type"]
de = pd.read_csv(A.figure("pseudobulk_de.csv.gz"))
stat = de.pivot_table(index="row", columns="gene", values="stat").fillna(0)    # Wald statistic, old vs young
robust = [r for r in stat.index if r in set(atlas.loc[atlas["robust"], "row"])]
tissue_of = dict(zip(atlas["row"], atlas["tissue"]))
R = ag.ROOT / "data" / "external" / "decoupler"
collectri = pd.read_csv(R / "collectri_mouse.csv")
progeny = pd.read_csv(R / "progeny_mouse.csv")
hallmark = pd.read_csv(R / "hallmark_mouse.csv")
print(len(stat), "rows tested in 09,", len(robust), "robust")

### 2. Transcription factor activity

For each row, every TF's activity is the t-value of a regression of the gene statistics on that TF's signed target weights. `tissues_up` / `tissues_down` count tissues with at least one robust row at t > 2 / < −2.

In [ ]:
def shared(es, rows, thr=2.0):
    e = es.loc[rows]
    up = (e > thr).groupby(e.index.map(tissue_of)).any().sum()
    down = (e < -thr).groupby(e.index.map(tissue_of)).any().sum()
    return pd.DataFrame({"mean_t": e.mean(), "rows_up": (e > thr).sum(), "rows_down": (e < -thr).sum(),
                         "tissues_up": up, "tissues_down": down}).sort_values("mean_t")

tf, tf_p = dc.mt.ulm(data=stat, net=collectri)
tf.round(3).to_csv(A.figure("mechanism_tf_activity.csv"))
tf_shared = shared(tf, robust)
tf_shared.to_csv(A.figure("mechanism_tf_shared.csv"))
print("up in robust rows of >= 4 tissues:")
print(tf_shared[tf_shared["tissues_up"] >= 4].sort_values("mean_t", ascending=False).round(2).head(20))
print("\ndown in robust rows of >= 4 tissues:")
print(tf_shared[tf_shared["tissues_down"] >= 4].round(2).head(20))

### 3. Which regulators move with the MHC class I rise?

MHC class I (*H2-D1*, *H2-K1*, *B2m*) rose in all 22 robust rows (09). Its known regulators: **Nlrc5** (the master transactivator of MHC class I), **Irf1** and **Stat1** (interferon-γ signaling), **Rela** / **Nfkb1** (NF-κB). **Ciita** drives MHC class II. If the rise is interferon- or NF-κB-driven, their activity should rise in the same rows.

**Circularity check.** A TF's activity is computed from its target genes, and the MHC class I genes are themselves targets of some of these regulators in CollecTRI (Ciita 14, Trp53 8, Rela 3, Nfkb1 2, Irf1 1, Stat1 1). A correlation between a regulator's activity and the MHC class I statistic would then be partly the MHC genes correlating with themselves. So activity is recomputed with every *H2-* gene and *B2m* removed from the network (`spearman_without_MHC_targets`). Either way this is co-variation across rows, not evidence that the regulator causes the rise.

In [ ]:
regulators = [g for g in ["Nlrc5", "Irf1", "Stat1", "Irf9", "Stat2", "Rela", "Nfkb1", "Ciita", "Trp53", "Nr3c1"] if g in tf.columns]
reg = tf.loc[robust, regulators]
mhc = stat.loc[robust, [g for g in ["H2-D1", "H2-K1", "B2m"] if g in stat.columns]].mean(axis=1).rename("MHC_I_stat")
corr = pd.concat([reg, mhc], axis=1).corr(method="spearman")["MHC_I_stat"].drop("MHC_I_stat")
net_wo = collectri[~collectri["target"].str.match(r"^(H2-|B2m$)")]
tf_wo, _ = dc.mt.ulm(data=stat.loc[robust], net=net_wo)
reg_wo = tf_wo[[g for g in regulators if g in tf_wo.columns]]
corr_wo = pd.concat([reg_wo, mhc], axis=1).corr(method="spearman")["MHC_I_stat"].drop("MHC_I_stat")
mhc_targets = collectri[collectri["target"].str.match(r"^(H2-|B2m$)")].groupby("source").size()
pd.DataFrame({"median_activity_robust": reg.median(), "rows_t_above_2": (reg > 2).sum(),
              "MHC_targets_in_net": mhc_targets.reindex(regulators).fillna(0).astype(int),
              "spearman_with_MHC_I_stat": corr, "spearman_without_MHC_targets": corr_wo}).round(2)

### 3b. Could the inflammatory signal be immune RNA floating into other cells?

Old tissues hold more immune cells, and some of their RNA ends up in every droplet (ambient RNA, 05b). A stromal or epithelial row could then show "rising" interferon and NF-κB targets that were never made by those cells. 09 marks, per row, the genes that are another cell type's markers present here only at an ambient level (`foreign`). Activities are recomputed with those genes removed per row; if the inflammatory regulators keep rising, the signal is the cells' own.

In [ ]:
own = de[~de["foreign"].astype(bool)].pivot_table(index="row", columns="gene", values="stat").fillna(0)
own_rows = [r for r in robust if r in own.index]
tf_own, _ = dc.mt.ulm(data=own.loc[own_rows], net=collectri)
hm_own, _ = dc.mt.ulm(data=own.loc[own_rows], net=hallmark)
hm_all, _ = dc.mt.ulm(data=stat.loc[own_rows], net=hallmark)
key_tf = [g for g in ["Nfkb1", "Rela", "Irf1", "Stat1", "Stat3", "Ciita"] if g in tf_own.columns]
key_hm = [h for h in hm_own.columns if any(k in h.upper() for k in ["INTERFERON", "INFLAMMATORY", "IL6", "COMPLEMENT", "TNFA"])]

def up_tissues(e):
    return (e > 2).groupby(e.index.map(tissue_of)).any().sum()

pd.concat([
    pd.DataFrame({"rows_up_all_genes": (tf.loc[own_rows, key_tf] > 2).sum(), "rows_up_own_genes": (tf_own[key_tf] > 2).sum(),
                  "tissues_up_all": up_tissues(tf.loc[own_rows, key_tf]), "tissues_up_own": up_tissues(tf_own[key_tf])}),
    pd.DataFrame({"rows_up_all_genes": (hm_all[key_hm] > 2).sum(), "rows_up_own_genes": (hm_own[key_hm] > 2).sum(),
                  "tissues_up_all": up_tissues(hm_all[key_hm]), "tissues_up_own": up_tissues(hm_own[key_hm])})])

### 4. Pathways (PROGENy) and hallmarks

Pathway activity per row; the heatmap shows robust rows only, grouped by compartment.

In [ ]:
pw, pw_p = dc.mt.mlm(data=stat, net=progeny)
pw.round(3).to_csv(A.figure("mechanism_pathway_activity.csv"))
hm, hm_p = dc.mt.ulm(data=stat, net=hallmark)
hm.round(3).to_csv(A.figure("mechanism_hallmark_activity.csv"))
print("PROGENy, robust rows:"); print(shared(pw, robust).round(2))
hs = shared(hm, robust)
hs.to_csv(A.figure("mechanism_hallmark_shared.csv"))
print("\nHallmarks moving the same way in >= 4 tissues:")
print(hs[(hs["tissues_up"] >= 4) | (hs["tissues_down"] >= 4)].sort_values("mean_t").round(2))

In [ ]:
comp = atlas.set_index("row")["compartment"]
order = sorted(robust, key=lambda r: (comp[r], r))
fig, ax = plt.subplots(figsize=(7, 0.32 * len(order) + 1.5))
sns.heatmap(pw.loc[order], cmap="RdBu_r", center=0, vmin=-6, vmax=6, ax=ax, cbar_kws={"label": "activity (t)"},
            yticklabels=[f"{r[:45]} [{comp[r]}]" for r in order])
ax.set_title("Pathway activity, old vs young (robust rows)")
fig.savefig(A.figure("mechanism_pathways.png"), dpi=200, bbox_inches="tight")

### 5. Batch-like regulators

Two regulators that are not about aging can still change between age groups: the **glucocorticoid receptor** (Nr3c1; stress hormone, and peaks with the circadian cycle) and **AP-1** (Fos, Jun: dissociation stress). 09 flagged *Tsc22d3*, a glucocorticoid target, falling in 5 tissues. If Nr3c1 activity falls with it, the old mice were likely sampled at a different hour or stress level, not aged in that respect.

In [ ]:
flags = [g for g in ["Nr3c1", "Fos", "Jun", "Junb", "Fosb", "Atf3", "Egr1", "Arntl", "Clock", "Nr1d1"] if g in tf.columns]
pd.DataFrame({"median_robust": tf.loc[robust, flags].median(), "rows_up": (tf.loc[robust, flags] > 2).sum(),
              "rows_down": (tf.loc[robust, flags] < -2).sum(),
              "tissues_down": (tf.loc[robust, flags] < -2).groupby(tf.loc[robust].index.map(tissue_of)).any().sum()}).round(2)

### 6. Cell–cell signals that change with age, mice as samples

For each tissue and each pair of rows tested in 09 (sender, receiver), and each ligand–receptor pair of liana's mouse consensus list: the interaction **rises** when the ligand rises in the sender and the receptor is expressed (baseMean > 20) in the receiver without falling; its score is the ligand's Wald statistic plus the receptor's. Pairs are kept when the ligand is significant (padj < 0.1) and not foreign to the sender. Within one tissue the rows share mice, so read the top list as candidates.

In [ ]:
lr = li.rs.select_resource("mouseconsensus")
lr = lr[~lr["ligand"].str.contains("_") & ~lr["receptor"].str.contains("_")]     # single-gene pairs only
d = de.set_index(["row", "gene"])
pairs = []
for tissue in sorted({tissue_of[r] for r in stat.index}):
    rows_t = [r for r in stat.index if tissue_of[r] == tissue]
    for src in rows_t:
        s = de[(de["row"] == src) & (de["padj"] < 0.1) & (de["log2FoldChange"] > 0) & ~de["foreign"]]
        lig = lr[lr["ligand"].isin(s["gene"])]
        if lig.empty:
            continue
        for tgt in rows_t:
            t = de[(de["row"] == tgt) & (de["baseMean"] > 20)].set_index("gene")
            ok = lig[lig["receptor"].isin(t.index)]
            for l, rcp in zip(ok["ligand"], ok["receptor"]):
                if t.loc[rcp, "stat"] < 0:
                    continue
                pairs.append({"tissue": tissue, "sender": src.split(" | ")[1], "receiver": tgt.split(" | ")[1],
                              "ligand": l, "receptor": rcp,
                              "ligand_log2fc": d.loc[(src, l), "log2FoldChange"],
                              "receptor_log2fc": t.loc[rcp, "log2FoldChange"],
                              "score": d.loc[(src, l), "stat"] + t.loc[rcp, "stat"],
                              "sender_robust": src in robust, "receiver_robust": tgt in robust})
lr_up = pd.DataFrame(pairs).sort_values("score", ascending=False)
lr_up.round(3).to_csv(A.figure("mechanism_ligand_receptor_up.csv"), index=False)
print(len(lr_up), "rising interactions;", lr_up.groupby(["ligand", "receptor"]).ngroups, "distinct pairs")
lr_up.head(25).round(2)

### 7. Ligands that rise across tissues

A ligand–receptor pair rising in 3+ tissues, and the complement and MHC class I families specifically.

In [ ]:
across = (lr_up.groupby(["ligand", "receptor"]).agg(tissues=("tissue", "nunique"), rows=("score", "size"),
                                                    best_score=("score", "max"),
                                                    where=("tissue", lambda v: ", ".join(sorted(set(v)))))
          .sort_values(["tissues", "best_score"], ascending=False))
across.to_csv(A.figure("mechanism_ligand_receptor_across.csv"))
print(across[across["tissues"] >= 3].head(25))
fam = lr_up[lr_up["ligand"].str.match(r"^(C3|C4b|Cfb|C1q|H2-|B2m)")]
print("\ncomplement and MHC class I ligands:")
fam.groupby(["ligand", "receptor"]).agg(tissues=("tissue", "nunique"), rows=("score", "size"),
                                        senders=("sender", lambda v: ", ".join(sorted(set(v))[:4]))).sort_values("tissues", ascending=False).head(20)

### Findings

22 robust rows from 7 tissues (and 33 others), DESeq2 statistics from 09 (mice as samples).

- **The shared aging signal co-varies with NF-κB and interferon activity.** Up in robust rows of 6 tissues: *Nfkb1* (18 rows, mean t 4.9), *Rela* (17), *Rel*; *Stat1* (16) and *Irf1* (18); also *Stat3*, *Hif1a*, *Spi1* and *Ciita* (the MHC class II driver). Nothing comparable falls: the strongest down-regulated TFs (*Bach2*, *Vhl*, *Smad6*) reach only 8–9 rows.
- **The MHC class I rise co-varies with them, and not by construction.** Across robust rows, the mean statistic of *H2-D1*/*H2-K1*/*B2m* correlates with *Irf1* activity (ρ 0.61), *Stat1* (0.58) and *Rela* (0.54). With every MHC gene removed from the target sets the correlations barely move (0.60, 0.56, 0.52); only *Ciita*, which has 14 MHC targets, drops (0.50 → 0.36). This is co-variation across rows, not proof that these regulators cause the rise. *Nlrc5*, the dedicated MHC class I transactivator, has no CollecTRI targets and could not be tested.
- **Pathways and hallmarks: inflammaging.** PROGENy JAK-STAT up (t 4.0), NF-κB and hypoxia up (2.2), TGF-β down in 8 rows. Hallmarks up in 6–7 tissues: interferon-α, IL-6/JAK/STAT3, inflammatory response, complement, p53, apoptosis, mTORC1 (t 3.0–4.1); down: epithelial–mesenchymal transition (matrix) and oxidative phosphorylation.
- **Not ambient immune RNA.** Recomputed without each row's foreign genes, the interferon and NF-κB regulators stay up in the same 6 tissues (*Irf1* 18 → 17 rows, *Stat1* 16 → 17, *Rela* 17 → 15, *Nfkb1* 18 → 13), as do the complement, IL-6/JAK/STAT3, interferon-γ and TNF-α hallmarks. The MHC class II driver *Ciita* halves (17 → 9 rows): part of the class II signal in non-immune rows is immune RNA, as the 09 ambient filter would predict.
- **Batch-like checks.** The glucocorticoid receptor (*Nr3c1*) does not change (2 of 22 rows), so the *Tsc22d3* fall of 09 is not a stress-hormone or time-of-day effect through this route. AP-1 does rise (*Jun* 15 rows, *Fos* 10, *Egr1* 12), and AP-1 is also the dissociation-stress response: part of the NF-κB/AP-1 signal may be how old tissue reacts to being dissociated, which this data cannot separate from in-vivo inflammation.
- **Signals between cells (candidates).** Rising in 4 tissues: *Tnf* → *Fas*, *Traf2*, *Notch1*; *Ccl5* → *Ccr1*, *Sdc4*; *Serpine2* → *Lrp1* (5). Stromal cells (bladder mesenchyme, heart fibroblast, muscle mesenchymal stem cell) send more complement *C3* to *Cd81* and *Lrp1* (3 tissues), and *H2-K1*/*B2m* rise toward T and NK cell receptors (*Cd8b1*, *Klrd1*, *Cd247*). These scores are built from the same ligand changes as 09, so they restate the MHC class I and complement rise rather than add independent evidence of communication. The top pair, *B2m*–*Hfe*, is an iron-regulation complex that the resource lists as a ligand–receptor pair, so it is not read as signaling.
- **Read as (exploratory):** the robust, replicating part of aging in this atlas co-varies with an interferon/NF-κB inflammatory program that raises antigen presentation and complement, plus a loss of matrix and oxidative metabolism genes.